# El Salvador Sintético — 02. Modelos econométricos, espaciales, NLP y supervivencia

**Autor:** Cristian Miranda · **Licencia:** CC-BY-4.0 (ver `LICENSE` en el repo) ·
**Repo:** https://github.com/MirandaCR/el-salvador-sintetico ·
**Dashboard en vivo:** https://mirandacr.github.io/el-salvador-sintetico/

Este notebook porta a Python los modelos que corren en Node.js en el repo
(`src/models.mjs`, `advanced.mjs`, `nlp.mjs`, `survival.mjs`, `highered.mjs`), usando librerías
estándar del ecosistema científico de Python (`statsmodels`, `scikit-learn`, `libpysal`/`esda`,
`spreg`, `mgwr`) en vez de las implementaciones manuales en JavaScript puro del pipeline original.

## ⚠️ Los datos son SINTÉTICOS

Estos modelos **ilustran método** (econometría de microdatos, econometría espacial, elección
discreta, supervivencia, NLP) sobre una población generada por IA — **no** son estimaciones
causales sobre El Salvador real. Ningún resultado aquí debe citarse como estadística oficial.

**Contenido:**
1. Carga de datos y agregados por departamento/municipio
2. OLS: años de escolaridad ~ edad + edad² + mujer + urbano + departamento
3. LPM: probabilidad de educación superior
4. Brechas crudas vs. condicionales (género, área)
5. Autocorrelación espacial: Moran's I global + LISA (departamentos y municipios)
6. Logit ordenado: nivel educativo (7 categorías)
7. Logit multinomial: sector ocupacional
8. Árbol de decisión (CART): educación superior
9. SAR (spatial lag) — municipios
10. GWR (regresión geográficamente ponderada) — municipios
11. k-means: tipología territorial de municipios
12. Monte Carlo: estabilidad del ranking municipal de ICH
13. NLP: términos distintivos por departamento y sector
14. Supervivencia (current-status): edad a la primera unión conyugal
15. Educación superior: sobreeducación, techo de cristal, concentración territorial


In [ ]:
# En Kaggle, la mayoría ya viene preinstalada. Lo que no, se instala aquí (con internet activado):
# !pip install -q datasets huggingface_hub libpysal esda mgwr spreg geopandas lifelines


In [ ]:
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 50)
np.random.seed(42)


## 1. Carga de datos y agregados

Mismo dataset que el notebook 01, cargado de forma independiente para que este notebook corra
solo.

In [ ]:
from datasets import load_dataset

ds = load_dataset("nvidia/Nemotron-Personas-El-Salvador")
df = ds["train"].to_pandas()

EDU_ORDER = ['ninguno', 'primaria', 'secundaria', 'bachillerato', 'tecnico', 'universitario', 'posgrado']
EDU_YEARS = {'ninguno': 0, 'primaria': 6, 'secundaria': 9, 'bachillerato': 12, 'tecnico': 14,
             'universitario': 16, 'posgrado': 18}
HIGHER = {'tecnico', 'universitario', 'posgrado'}
DEPTS = ['San Salvador', 'La Libertad', 'Santa Ana', 'Sonsonate', 'San Miguel', 'Ahuachapán',
         'Usulután', 'La Paz', 'Cuscatlán', 'La Unión', 'Chalatenango', 'Morazán', 'San Vicente', 'Cabañas']
REF_DEPT = 'San Salvador'

df["age"] = df["age"].astype(int)
df["education_level"] = pd.Categorical(df["education_level"], categories=EDU_ORDER, ordered=True)
df["eduYears"] = df["education_level"].map(EDU_YEARS).astype(float)  # .astype: map() on Categorical keeps categorical dtype otherwise
df["higher"] = df["education_level"].isin(HIGHER).astype(int)
df["female"] = (df["sex"] == "Femenino").astype(int)
df["urban"] = (df["area"] == "urbano").astype(int)
df["department"] = pd.Categorical(df["department"], categories=DEPTS)
print(df.shape)


### Sectores ocupacionales

Igual que en `advanced.mjs`/`nlp.mjs`: mapeamos la ocupación (texto libre por actividad económica)
a uno de 6 sectores agregados mediante keywords.

In [ ]:
SECTORS = ['Agro', 'Comercio', 'Comida y hogar', 'Industria y construcción', 'Transporte', 'Servicios y Estado']

def norm(s):
    import unicodedata
    s = str(s).lower()
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def sector_of(occ):
    o = norm(occ)
    if re.search('cultivo|agricol|ganad|pesca|silvicult|pecuari|cafe|forestal', o): return 0
    if re.search('restaurante|comida|hogares como|alojamiento|bebidas|hospeda', o): return 2
    if re.search('transporte|almacenamiento|mensajeri|correo|logistic', o): return 4
    if re.search('fabricacion|elaboracion|construccion|prendas|confeccion|textil|manufactur|instalacion|reparacion|panaderia|metal|madera|calzado', o): return 3
    if re.search('administracion publica|ensenanza|educacion|salud|hospital|financ|seguro|juridic|contab|profesional|cientific|tecnic|informacion|telecomunic|inmobili|publicidad|consultor', o): return 5
    if re.search('venta al por menor|venta al por mayor|comercio|reparacion de vehiculos', o): return 1
    return 5

df["sector"] = df["occupation"].apply(sector_of)
df["sector"].value_counts()


## 2. OLS: años de escolaridad ~ edad + edad² + mujer + urbano + departamento

Referencia: San Salvador. Réplica de `models.mjs` (que hacía la acumulación X'X/X'y a mano);
aquí usamos `statsmodels` directamente.

In [ ]:
ols_edu = smf.ols(
    "eduYears ~ age + I(age**2) + female + urban + "
    f"C(department, Treatment(reference='{REF_DEPT}'))",
    data=df
).fit()
print(f"R² = {ols_edu.rsquared:.4f}   n = {int(ols_edu.nobs)}")
ols_edu.summary2().tables[1].head(8)


## 3. LPM: probabilidad de educación superior

Modelo de probabilidad lineal (mismas covariables) para P(educación técnica/universitaria/posgrado).

In [ ]:
lpm_higher = smf.ols(
    "higher ~ age + I(age**2) + female + urban + "
    f"C(department, Treatment(reference='{REF_DEPT}'))",
    data=df
).fit()
print(f"R² = {lpm_higher.rsquared:.4f}")
lpm_higher.summary2().tables[1].head(5)


## 4. Brechas crudas vs. condicionales

In [ ]:
gap_gender_raw = df.loc[df.female == 1, "eduYears"].mean() - df.loc[df.female == 0, "eduYears"].mean()
gap_area_raw = df.loc[df.urban == 1, "eduYears"].mean() - df.loc[df.urban == 0, "eduYears"].mean()
gap_gender_cond = ols_edu.params.get("female")
gap_area_cond = ols_edu.params.get("urban")

print(f"Género   — cruda: {gap_gender_raw:+.3f}  |  condicional (OLS): {gap_gender_cond:+.3f}  (p={ols_edu.pvalues['female']:.4f})")
print(f"Urbano   — cruda: {gap_area_raw:+.3f}  |  condicional (OLS): {gap_area_cond:+.3f}  (p={ols_edu.pvalues['urban']:.4f})")


## 5. Autocorrelación espacial: Moran's I + LISA

Reutilizamos la geometría de departamentos/municipios ya procesada y publicada en el repo
(`dashboard/data/*.geojson`) — misma fuente que usa el dashboard, para que el análisis sea
consistente en ambos lugares.

In [ ]:
import geopandas as gpd
import libpysal
from esda.moran import Moran, Moran_Local

GEO_BASE = "https://raw.githubusercontent.com/MirandaCR/el-salvador-sintetico/main/dashboard/data"
gdf_dept_geo = gpd.read_file(f"{GEO_BASE}/departments.geojson")
gdf_muni_geo = gpd.read_file(f"{GEO_BASE}/municipios.geojson")
centroids = json.loads(pd.io.common.urlopen(f"{GEO_BASE}/muni_centroids.json").read())
print(gdf_dept_geo.shape, gdf_muni_geo.shape, len(centroids))


In [ ]:
def minmax(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) if mx > mn else pd.Series(0.5, index=s.index)

def build_agg(group_col):
    g = df.groupby(group_col, observed=True).agg(
        count=("age", "size"),
        pctUrban=("urban", lambda s: 100 * s.mean()),
        eduYears=("eduYears", "mean"),
        eduStd=("eduYears", "std"),
        pctHigher=("higher", lambda s: 100 * s.mean()),
        meanAge=("age", "mean"),
    ).reset_index().rename(columns={group_col: "name"})
    g["ich"] = (100 * (0.5 * minmax(g["eduYears"]) + 0.3 * minmax(g["pctHigher"]) + 0.2 * minmax(g["pctUrban"]))).round(1)
    return g

dept_agg = build_agg("department")
muni_agg = build_agg("municipality")
print(dept_agg.shape, muni_agg.shape)


In [ ]:
gdf_dept = gdf_dept_geo.merge(dept_agg, left_on="department", right_on="name", how="inner")
w_dept = libpysal.weights.Queen.from_dataframe(gdf_dept, use_index=False)
w_dept.transform = "r"

moran_dept = Moran(gdf_dept["ich"].values, w_dept, permutations=999)
lisa_dept = Moran_Local(gdf_dept["ich"].values, w_dept, permutations=999)
print(f"Moran's I (ICH, 14 deptos) = {moran_dept.I:.4f}   E[I] = {moran_dept.EI:.4f}   p_sim = {moran_dept.p_sim:.3f}"
      f"   {'*** significativo' if moran_dept.p_sim < 0.05 else '(no significativo)'}")

gdf_muni = gdf_muni_geo.merge(muni_agg, left_on="municipality", right_on="name", how="inner")
w_muni = libpysal.weights.Queen.from_dataframe(gdf_muni, use_index=False)
w_muni.transform = "r"
moran_muni = Moran(gdf_muni["ich"].values, w_muni, permutations=999)
print(f"Moran's I (ICH, {len(gdf_muni)} municipios) = {moran_muni.I:.4f}   E[I] = {moran_muni.EI:.4f}   p_sim = {moran_muni.p_sim:.3f}"
      f"   {'*** significativo' if moran_muni.p_sim < 0.05 else '(no significativo)'}")


In [ ]:
QUAD = {1: "Alto-Alto", 2: "Bajo-Bajo", 3: "Alto-Bajo", 4: "Bajo-Alto"}
gdf_dept["lisa_quadrant"] = [QUAD[q] for q in lisa_dept.q]
gdf_dept["lisa_p"] = lisa_dept.p_sim
gdf_dept["lisa_sig"] = gdf_dept["lisa_p"] < 0.05

fig, ax = plt.subplots(figsize=(7, 7))
gdf_dept.plot(column="ich", cmap="Blues", legend=True, edgecolor="white", ax=ax)
ax.set_title("ICH por departamento (mapa coroplético)")
ax.axis("off")
plt.tight_layout()
plt.show()

gdf_dept[["department", "ich", "lisa_quadrant", "lisa_p", "lisa_sig"]].sort_values("ich", ascending=False)


## 6. Logit ordenado: nivel educativo (7 categorías)

Usamos `OrderedModel` de `statsmodels` (logit ordenado), equivalente al MLE por Adam que hace
`advanced.mjs` a mano.

In [ ]:
dept_dummies = pd.get_dummies(df["department"], prefix="dep", drop_first=False)
dept_dummies = dept_dummies.drop(columns=[f"dep_{REF_DEPT}"])
X_ord = pd.concat([df[["age", "female", "urban"]], df["age"].pow(2).rename("age2"), dept_dummies], axis=1).astype(float)

mod_ord = OrderedModel(df["education_level"], X_ord, distr="logit")
res_ord = mod_ord.fit(method="bfgs", disp=False, maxiter=300)

or_urban = np.exp(res_ord.params["urban"])
or_female = np.exp(res_ord.params["female"])
print(f"Odds ratio urbano = {or_urban:.2f}   Odds ratio mujer = {or_female:.2f}")
res_ord.params[["age", "age2", "female", "urban"]]


## 7. Logit multinomial: sector ocupacional

Elección discreta del sector (6 categorías) como función de edad/sexo/área — microeconomía de
utilidad aleatoria (McFadden).

In [ ]:
X_mn = sm.add_constant(df[["age", "female", "urban"]].astype(float))
mod_mn = sm.MNLogit(df["sector"], X_mn)
res_mn = mod_mn.fit(method="newton", disp=False, maxiter=100)

base_prob = res_mn.predict(X_mn).mean(axis=0)
print("Probabilidad promedio por sector:")
for i, s in enumerate(SECTORS[1:], start=1):  # sector 0 (Agro) es la categoría base en MNLogit
    print(f"  {s}: {100*base_prob[i]:.1f}%")

# efecto marginal de "urbano" (urbano=1 vs urbano=0), manteniendo el resto en su media
margeff = res_mn.get_margeff(at="mean")
print()
print(margeff.summary())


## 8. Árbol de decisión (CART): educación superior

Features interpretables (edad, sexo, área, San Salvador, La Libertad) — igual que `advanced.mjs`.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split

Xt = pd.DataFrame({
    "edad": df["age"],
    "mujer": df["female"],
    "urbano": df["urban"],
    "depSanSalvador": (df["department"] == "San Salvador").astype(int),
    "depLaLibertad": (df["department"] == "La Libertad").astype(int),
})
yt = df["higher"]
Xtr, Xte, ytr, yte = train_test_split(Xt, yt, test_size=0.2, random_state=42, stratify=yt)

tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=2000, random_state=42).fit(Xtr, ytr)
acc = tree.score(Xte, yte)
base_rate = yt.mean()
print(f"Accuracy = {acc:.3f}   (base rate = {base_rate:.3f})")

importances = pd.Series(tree.feature_importances_, index=Xt.columns).sort_values(ascending=False)
print(importances)

plt.figure(figsize=(16, 6))
plot_tree(tree, feature_names=Xt.columns, class_names=["no_superior", "superior"], filled=True, fontsize=8)
plt.show()


## 9. SAR (spatial lag) — municipios

`ICH = ρ·W·ICH + Xβ + ε`, con matriz de pesos k-vecinos-más-cercanos (k=5) sobre los centroides
municipales. Usamos `spreg.ML_Lag` (PySAL) en vez del grid-search manual sobre ρ del JS.

In [ ]:
from spreg import ML_Lag

muni_agg_c = muni_agg[muni_agg["name"].isin(centroids.keys())].copy()
coords_m = np.array([centroids[n] for n in muni_agg_c["name"]])  # [lon, lat]

w_knn = libpysal.weights.KNN.from_array(coords_m, k=5)
w_knn.transform = "r"

y_sar = muni_agg_c[["ich"]].values
X_sar = muni_agg_c[["pctUrban", "pctHigher"]].values
sar = ML_Lag(y_sar, X_sar, w=w_knn, name_y="ich", name_x=["pctUrban", "pctHigher"])

print(f"rho (ρ) = {sar.rho:.3f}")
print("Interpretación:", "positivo -> derrame espacial (el ICH de un municipio se 'arrastra' por sus vecinos)"
      if sar.rho > 0 else "sin arrastre espacial claro")
print(pd.DataFrame(sar.betas, index=["const"] + ["pctUrban", "pctHigher"], columns=["beta"]))


## 10. GWR (regresión geográficamente ponderada) — municipios

Coeficiente LOCAL de `eduYears ~ pctUrban` por municipio (kernel gaussiano, ancho de banda óptimo
por validación cruzada) — a diferencia del SAR/OLS, aquí el efecto de "% urbano" puede variar
según el territorio.

In [ ]:
from mgwr.gwr import GWR
from mgwr.sel_bw import Sel_BW

y_gwr = muni_agg_c[["eduYears"]].values
X_gwr = muni_agg_c[["pctUrban"]].values

bw = Sel_BW(coords_m, y_gwr, X_gwr).search()
gwr_res = GWR(coords_m, y_gwr, X_gwr, bw).fit()

muni_agg_c["gwr_slope_pctUrban"] = gwr_res.params[:, 1]
print(f"Ancho de banda seleccionado: {bw:.1f} vecinos")
print(f"Rango de la pendiente local: [{muni_agg_c['gwr_slope_pctUrban'].min():.4f}, {muni_agg_c['gwr_slope_pctUrban'].max():.4f}]")
muni_agg_c[["name", "gwr_slope_pctUrban"]].sort_values("gwr_slope_pctUrban", ascending=False).head(10)


## 11. k-means: tipología territorial de municipios

Cuatro perfiles (bajo → alto capital humano), etiquetados por el ICH medio de cada cluster.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

feat_cols = ["eduYears", "pctUrban", "pctHigher", "meanAge"]
Z = StandardScaler().fit_transform(muni_agg_c[feat_cols])
km = KMeans(n_clusters=4, random_state=42, n_init=10).fit(Z)
muni_agg_c["cluster_raw"] = km.labels_

cluster_ich = muni_agg_c.groupby("cluster_raw")["ich"].mean().sort_values()
labels_order = ["Rezago rural", "En desarrollo", "Emergente urbano", "Núcleo desarrollado"]
label_map = {c: labels_order[i] for i, c in enumerate(cluster_ich.index)}
muni_agg_c["cluster"] = muni_agg_c["cluster_raw"].map(label_map)

summary = muni_agg_c.groupby("cluster")["ich"].agg(["size", "mean"]).round(1)
summary


## 12. Monte Carlo: estabilidad del ranking municipal de ICH

Simulamos incertidumbre en la escolaridad media (usando el error estándar `eduStd/√n` por
municipio) y vemos qué tan estable es el ranking resultante.

In [ ]:
T = 1000
names = muni_agg_c["name"].values
n = len(names)
se = (muni_agg_c["eduStd"] / np.sqrt(muni_agg_c["count"])).values
base = muni_agg_c["eduYears"].values

rank_positions = np.zeros((n, n))  # [municipio, posición_de_ranking] -> conteo
rng = np.random.default_rng(42)
for _ in range(T):
    sim = base + se * rng.standard_normal(n)
    order = np.argsort(-sim)  # de mayor a menor
    for pos, idx in enumerate(order):
        rank_positions[idx, pos] += 1

avg_rank = (rank_positions * np.arange(1, n + 1)).sum(axis=1) / T
prob_top5 = rank_positions[:, :5].sum(axis=1) / T

mc = pd.DataFrame({"name": names, "eduYears": base, "rankProm": avg_rank.round(1), "probTop5_%": (100 * prob_top5).round(0)})
mc.sort_values("rankProm").head(10)


## 13. NLP: términos distintivos por departamento y sector

Score de distintividad = (frecuencia relativa dentro del grupo) / (frecuencia relativa global),
igual que `nlp.mjs` — qué palabras "sobre-representa" cada departamento en sus metas de carrera
o trasfondo cultural, respecto al promedio nacional.

In [ ]:
from collections import Counter

STOP = set(('de la el en y a los las del un una con por para su sus al lo se que como mas más o e u '
            'ni ademas además tambien también sobre entre desde hasta muy sin le les mi tu es son ser '
            'estar cada donde cuando quien cual todo toda todos todas otro otra este esta ese esa esos '
            'esas hay ha he han ya pero porque si no me te nos vez veces gran grande mismo misma solo '
            'sola tras cabe so aunque mientras segun según tan tanto poco mucha mucho muchos muchas '
            'algun alguna algunos algunas cosa cosas persona personas gente').split())

def tok(text):
    text = str(text or "").lower()
    text = re.sub(r"[^a-záéíóúñü\s]", " ", text)
    return [w for w in text.split() if len(w) > 3 and w not in STOP]

# reusamos las columnas de texto libre (career_goals_and_ambitions, cultural_background)
ds_text = ds["train"].select_columns(["department", "occupation", "career_goals_and_ambitions", "cultural_background"]).to_pandas()
ds_text["sector"] = ds_text["occupation"].apply(sector_of)

global_counter = Counter()
dept_counters = {}
for d, sub in ds_text.groupby("department"):
    c = Counter()
    for text in sub["cultural_background"]:
        c.update(tok(text))
    dept_counters[d] = c
    global_counter.update(c)
global_tot = sum(global_counter.values())

def distinctive(counter, min_c=40, n=8):
    tot = sum(counter.values())
    scored = [(w, round((c / tot) / (global_counter[w] / global_tot), 2), c)
              for w, c in counter.items() if c >= min_c and global_counter[w]]
    return sorted(scored, key=lambda x: -x[1])[:n]

for d in ["San Salvador", "La Unión"]:
    print(f"{d}: {distinctive(dept_counters[d])}")


## 14. Supervivencia (current-status): edad a la primera unión conyugal

**Nota metodológica doble:** (1) dato sintético, (2) diseño *current-status* — es una foto
transversal, no un seguimiento longitudinal, así que aproximamos la curva de "supervivencia de
la soltería" S(edad) = % que sigue soltero a esa edad, sin fechas de evento reales.

In [ ]:
AMIN, AMAX = 18, 80
dsurv = df[(df["age"] >= AMIN) & (df["age"] <= AMAX)].copy()
dsurv["single"] = (dsurv["marital_status"] == "soltero").astype(int)
dsurv["partnered"] = 1 - dsurv["single"]

def survival_curve(sub):
    s = sub.groupby("age")["single"].mean().reindex(range(AMIN, AMAX + 1))
    return s.rolling(3, center=True, min_periods=1).mean()

def median_age(sub):
    s = sub.groupby("age")["single"].mean().reindex(range(AMIN, AMAX + 1))
    prev = None
    for a, v in s.items():
        if pd.isna(v):
            continue
        if v <= 0.5:
            if prev is not None and prev[1] > 0.5:
                frac = (prev[1] - 0.5) / (prev[1] - v)
                return round(prev[0] + frac * (a - prev[0]), 1)
            return a
        prev = (a, v)
    return None

groups = {
    "Todos": dsurv, "Hombres": dsurv[dsurv.female == 0], "Mujeres": dsurv[dsurv.female == 1],
    "Urbano": dsurv[dsurv.urban == 1], "Rural": dsurv[dsurv.urban == 0],
    "Educ. superior": dsurv[dsurv.higher == 1], "Sin educ. superior": dsurv[dsurv.higher == 0],
}
medians = {k: median_age(v) for k, v in groups.items()}
print("Mediana de edad a la primera unión, por grupo:")
for k, v in medians.items():
    print(f"  {k}: {v}")

fig, ax = plt.subplots(figsize=(9, 6))
for k, v in groups.items():
    survival_curve(v).plot(ax=ax, label=k)
ax.set_ylabel("S(edad) = % soltero")
ax.set_xlabel("Edad")
ax.set_title("Curvas de supervivencia de la soltería (current-status)")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
ageZ = (dsurv["age"] - dsurv["age"].mean()) / dsurv["age"].std()
X_surv = sm.add_constant(pd.DataFrame({
    "age_z": ageZ, "age_z2": ageZ ** 2, "female": dsurv["female"], "urban": dsurv["urban"], "higher": dsurv["higher"],
}))
logit_surv = sm.Logit(dsurv["partnered"], X_surv).fit(disp=False)
odds = np.exp(logit_surv.params)
print("Odds ratios (partnered ~ edad + edad² + mujer + urbano + superior):")
print(odds)


## 15. Educación superior: sobreeducación, techo de cristal, concentración territorial

Réplica de `highered.mjs`: (A) % de universitarios/posgrado en ocupaciones que NO son intensivas
en conocimiento, (B) embudo educativo + techo de cristal (composición por sexo en cada nivel),
(C) concentración territorial del posgrado (índice HHI).

In [ ]:
def norm2(s):
    import unicodedata
    return ''.join(c for c in unicodedata.normalize('NFD', str(s).lower()) if unicodedata.category(c) != 'Mn')

def is_high_skill(occ):
    return bool(re.search(
        'ensen|educaci|hospital|salud|medic|enfermer|financ|banc|seguro|jurid|abogac|contab|'
        'cientif|ingenier|arquitect|profesional|consultor|informacion|telecomunic|program|'
        'software|investigaci|universidad|contabilidad|auditor', norm2(occ)))

df_he = df.copy()
df_he["occupation_raw"] = ds["train"].to_pandas()["occupation"]  # (ya está en df, alias por claridad)
df_he["is_high_skill"] = df_he["occupation"].apply(is_high_skill)

univ_posgrado = df_he[df_he["education_level"].isin(["universitario", "posgrado"])]
overall_rate = round(100 * (~univ_posgrado["is_high_skill"]).mean(), 1)
by_sex = round(100 * univ_posgrado.groupby("sex")["is_high_skill"].apply(lambda s: (~s).mean()), 1)
by_area = round(100 * univ_posgrado.groupby("area")["is_high_skill"].apply(lambda s: (~s).mean()), 1)
print(f"Sobreeducación (universitarios+posgrado en ocupación NO intensiva en conocimiento): {overall_rate}%")
print(by_sex)
print(by_area)


In [ ]:
# B) embudo educativo + techo de cristal
funnel = df_he["education_level"].value_counts().reindex(EDU_ORDER).rename("count").to_frame()
funnel["pctOfAdults"] = round(100 * funnel["count"] / len(df_he), 1)
funnel["pctFemale"] = round(100 * df_he.groupby("education_level", observed=True)["female"].mean(), 1)
print(funnel)

glass_ceiling = {
    "pctFemaleBachillerato": funnel.loc["bachillerato", "pctFemale"],
    "pctFemaleUniversitario": funnel.loc["universitario", "pctFemale"],
    "pctFemalePosgrado": funnel.loc["posgrado", "pctFemale"],
}
print()
print("Techo de cristal (% mujeres por nivel):", glass_ceiling)


In [ ]:
# C) concentración territorial del posgrado (HHI)
dept_pos = df_he.groupby("department", observed=True).agg(
    n=("age", "size"),
    tertiary=("higher", "sum"),
    posgrado=("education_level", lambda s: (s == "posgrado").sum()),
).reset_index()
dept_pos["tertiaryRate"] = round(100 * dept_pos["tertiary"] / dept_pos["n"], 1)
tot_pos = dept_pos["posgrado"].sum()
dept_pos["shareOfPosgrado"] = round(100 * dept_pos["posgrado"] / tot_pos, 1)
dept_pos = dept_pos.sort_values("shareOfPosgrado", ascending=False)

hhi = round((dept_pos["shareOfPosgrado"] ** 2).sum(), 0)
print(f"HHI de concentración del posgrado entre departamentos: {hhi} (0-10000; más alto = más concentrado)")
print(f"% del posgrado nacional en el top-3 departamentos: {dept_pos['shareOfPosgrado'].head(3).sum():.1f}%")
dept_pos[["department", "tertiaryRate", "shareOfPosgrado"]]


## Conclusiones y límites

- Todos los modelos (OLS, LPM, Moran's I/LISA, logit ordenado/multinomial, CART, SAR, GWR,
  k-means, Monte Carlo, NLP distintivo, supervivencia, sobreeducación/HHI) replican en Python
  la analítica del pipeline de Node.js del repo, usando librerías estándar de econometría
  (`statsmodels`), machine learning (`scikit-learn`) y econometría espacial (`libpysal`, `esda`,
  `spreg`, `mgwr`) en vez de las implementaciones manuales en JavaScript.
- **Todo es sobre datos sintéticos** generados por un LLM calibrado a distribuciones demográficas
  — los coeficientes, odds ratios y clusters aquí **no** son efectos causales reales, son un
  ejercicio de método.
- Ver el notebook `01_eda_estadistica_descriptiva.ipynb` para el procesamiento de datos y la
  estadística descriptiva previa a estos modelos.

**Repo:** https://github.com/MirandaCR/el-salvador-sintetico ·
**Dashboard:** https://mirandacr.github.io/el-salvador-sintetico/ ·
**Licencia:** CC-BY-4.0 — atribución a Cristian Miranda.
